In [1]:
import torch.optim as optim
import matplotlib.pyplot as plt

from src.load_and_save import save_model
from src.model import SimpleNN
from src.pruning import get_intermediate_outputs_as_numpy
from src.training import get_accuracy
from src.utils import device
from settings import settings
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torch.nn as nn
import torch

from src.pruning import is_all_layers_separated
import numpy as np
from pathlib import Path



In [2]:
# Define the number of classes in MNIST (digits 0-9)
num_classes = 10

# Define the transformation to apply to the images
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert images to PyTorch tensors
    transforms.Lambda(lambda x: x.view(-1))  # Flatten the images to a single dimension
])

# Custom transform to one-hot encode the labels
class OneHotEncode:
    def __init__(self, num_classes):
        self.num_classes = num_classes

    def __call__(self, label):
        return torch.eye(self.num_classes)[label]

# Load the full training dataset
full_train_dataset = datasets.MNIST(
    root=settings.data_path,
    train=True,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Split the full training dataset into training and validation datasets
train_size = int(0.8 * len(full_train_dataset))  # 80% for training
val_size = len(full_train_dataset) - train_size  # 20% for validation
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Load the test dataset
test_dataset = datasets.MNIST(
    root=settings.data_path,
    train=False,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Create DataLoaders for training, validation, and test sets
train_dataloader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=2048, shuffle=False)

In [3]:
from src.model import OneScrambleLayerNN
import pandas as pd

# Create the DataFrame:
columns = ['epoch', 'scramble_distance', 'separation', 'accuracy']
data = []
df = pd.DataFrame(data, columns=columns)

# Get test data:
test_data, _ = next(iter(test_dataloader))
test_data = test_data.to(device)

def get_quantile_of_separation(intermediate_output: list[float], quantile: float) -> float:
    return np.quantile(abs(np.array(intermediate_output)), quantile)

def get_average_separation(test_data, model) -> float:
    intermediate_layer = model.layer1(test_data)
    all_quantiles = []
    for i in range(intermediate_layer.shape[1]):
        intermediate_output = intermediate_layer[:,i].tolist()
        all_quantiles.append(get_quantile_of_separation(intermediate_output, 0.05))
    return np.mean(all_quantiles)

num_epochs: int = 500
for scramble_distance in [0.5]:

    # Instantiate the model
    model = OneScrambleLayerNN(784, 10, scramble_distance).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=float(1e-2))

    # Training loop
    for epoch in range(num_epochs):
        for inputs, labels in train_dataloader:
            if epoch == 0:
                break
            # Forward pass
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()


        # Add data to dataframe:
        separation: float = get_average_separation(test_data, model)
        accuracy: float = float(get_accuracy(model, val_dataloader))
        df.loc[len(df)] = [epoch, scramble_distance, separation, accuracy]
        print(df.tail(1))
    df.to_csv("experiment.csv")
    break


   epoch  scramble_distance  separation  accuracy
0    0.0                0.5    0.100368  0.101667
   epoch  scramble_distance  separation  accuracy
1    1.0                0.5    0.250156    0.8955
   epoch  scramble_distance  separation  accuracy
2    2.0                0.5    0.261701  0.920333
   epoch  scramble_distance  separation  accuracy
3    3.0                0.5    0.268016  0.931833
   epoch  scramble_distance  separation  accuracy
4    4.0                0.5    0.271028  0.937167
   epoch  scramble_distance  separation  accuracy
5    5.0                0.5    0.262685  0.937583
   epoch  scramble_distance  separation  accuracy
6    6.0                0.5    0.277035  0.940167
   epoch  scramble_distance  separation  accuracy
7    7.0                0.5    0.266694   0.94575
   epoch  scramble_distance  separation  accuracy
8    8.0                0.5    0.288213  0.946583
   epoch  scramble_distance  separation  accuracy
9    9.0                0.5    0.291278  0.945167


In [4]:

df.to_csv(Path(__file__).parent / "experiment.csv")


NameError: name '__file__' is not defined